# Incident Response Runbook: Privilege Escalation - Abuse Elevation Control Mechanism

**Tactic:** Privilege Escalation
**Technique:** Abuse Elevation Control Mechanism (T1548)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Abuse Elevation Control Mechanism activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add project root to path for imports
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
privilege_escalation_indicators = []
incident_id = None

# Query Splunk for privilege escalation indicators
print(f"\n[DETECTION] Querying Splunk for privilege escalation activities...")
try:
    # UAC bypass and elevation abuse
    uac_bypass_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4625 OR EventCode=4672 OR EventCode=4688
    | eval risk_score = case(
        match(CommandLine, "bypassuac|uac.*bypass|elevated.*privilege"), 10,
        match(CommandLine, "runas.*admin|sudo.*-s"), 9,
        match(CommandLine, "psexec.*-s|powershell.*-executionpolicy.*bypass.*admin"), 10,
        match(CommandLine, "schtasks.*highest|at.*interactive"), 8,
        match(CommandLine, "net.*localgroup.*administrators"), 9,
        match(CommandLine, "reg.*add.*hkcu.*software.*classes.*mscfile"), 10,
        match(CommandLine, "cmstp.*inf|makecab.*cab"), 9,
        match(CommandLine, "msiexec.*alwaysinstall"), 8,
        1==1, 4
    )
    | where risk_score >= 8
    | stats count by host, user, CommandLine, risk_score
    | sort -risk_score
    """

    uac_results = splunk.search(uac_bypass_query, timeframe="-24h")
    if uac_results:
        print(f"   Found {len(uac_results)} UAC bypass/elevation abuse attempts")
        for result in uac_results[:10]:  # Top 10
            privilege_escalation_indicators.append({
                'type': 'uac_bypass_elevation_abuse',
                'host': result.get('host'),
                'user': result.get('user'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

    # Token manipulation and theft
    token_manip_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4672 OR EventCode=4688
    | eval risk_score = case(
        match(CommandLine, "token.*steal|token.*impersonate|duplicate.*token"), 10,
        match(CommandLine, "seimpersonateprivilege|seassignprimarytokenprivilege"), 9,
        match(CommandLine, "incognito|tokenvator|mimikatz.*sekurlsa"), 10,
        match(CommandLine, "rundll32.*comsvcs.*minidump"), 9,
        match(CommandLine, "procdump.*lsass|taskmgr.*lsass"), 8,
        1==1, 4
    )
    | where risk_score >= 8
    | stats count by host, user, CommandLine, risk_score
    | sort -risk_score
    """

    token_results = splunk.search(token_manip_query, timeframe="-24h")
    if token_results:
        print(f"   Found {len(token_results)} token manipulation attempts")
        for result in token_results[:10]:  # Top 10
            privilege_escalation_indicators.append({
                'type': 'token_manipulation',
                'host': result.get('host'),
                'user': result.get('user'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

    # Process injection for privilege escalation
    injection_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4688
    | eval risk_score = case(
        match(CommandLine, "inject.*dll|reflective.*loader|process.*hollow"), 9,
        match(CommandLine, "rundll32.*javascript:|regsvr32.*scrobj"), 8,
        match(CommandLine, "powershell.*reflective|empire.*agent"), 10,
        match(CommandLine, "cobalt.*strike|metasploit.*meterpreter"), 10,
        1==1, 4
    )
    | where risk_score >= 8
    | stats count by host, user, CommandLine, risk_score
    | sort -risk_score
    """

    injection_results = splunk.search(injection_query, timeframe="-24h")
    if injection_results:
        print(f"   Found {len(injection_results)} process injection attempts")
        for result in injection_results[:10]:  # Top 10
            privilege_escalation_indicators.append({
                'type': 'process_injection_privilege_escalation',
                'host': result.get('host'),
                'user': result.get('user'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

except Exception as e:
    print(f"   Splunk query failed: {e}")

# Get unique affected systems/users
unique_hosts = set()
unique_users = set()
for indicator in privilege_escalation_indicators:
    if indicator.get('host'):
        unique_hosts.add(indicator['host'])
    if indicator.get('user'):
        unique_users.add(indicator['user'])

# Check CrowdStrike for endpoint details
print(f"\n[DETECTION] Checking CrowdStrike for endpoint details...")
for host in unique_hosts:
    try:
        cs_info = crowdstrike.get_host_info(host)
        if cs_info:
            affected_systems.append({
                'hostname': host,
                'device_id': cs_info.get('device_id'),
                'os': cs_info.get('os_version'),
                'last_seen': cs_info.get('last_seen'),
                'activities': [a for a in privilege_escalation_indicators if a.get('host') == host]
            })
            print(f"   Found endpoint details for: {host}")
    except Exception as e:
        print(f"   CrowdStrike check failed for {host}: {e}")

# Enrich with threat intelligence
print(f"\n[INTEL] Enriching with threat intelligence...")
for indicator in privilege_escalation_indicators[:5]:  # Check top 5
    try:
        if indicator.get('command'):
            # Extract suspicious file names or commands
            suspicious_patterns = re.findall(r'([a-zA-Z0-9\-_]+\.(exe|dll|bat|ps1|vbs|js))', indicator['command'], re.IGNORECASE)
            for file_name, ext in suspicious_patterns:
                full_name = f"{file_name}.{ext}"
                ti_data = misp.lookup_filename(full_name)
                if ti_data:
                    indicator['threat_intel'] = ti_data
                    print(f"   Threat intel found for file: {full_name}")
                    break

            # Check for known privilege escalation tools
            tool_names = ['mimikatz', 'incognito', 'tokenvator', 'psexec', 'cobalt', 'metasploit', 'empire']
            for tool in tool_names:
                if tool.lower() in indicator['command'].lower():
                    ti_data = misp.lookup_filename(tool)
                    if ti_data:
                        indicator['threat_intel'] = ti_data
                        print(f"   Threat intel found for tool: {tool}")
                        break
    except Exception as e:
        print(f"   Threat intelligence lookup failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    case_data = {
        'title': f'Privilege Escalation Activities - {len(privilege_escalation_indicators)} indicators',
        'description': f'Detected privilege escalation attempts affecting {len(unique_users)} users across {len(affected_systems)} systems',
        'severity': 'CRITICAL',
        'tactic': 'Privilege Escalation',
        'technique': 'Abuse Elevation Control Mechanism (T1548)',
        'indicators': privilege_escalation_indicators[:10],  # Top 10 indicators
        'detection_time': detection_time
    }
    incident_id = iris.create_case(case_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(privilege_escalation_indicators)} privilege escalation indicators detected")
print(f"  - {len(unique_users)} affected users identified")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - Threat intelligence enriched for {len([i for i in privilege_escalation_indicators if i.get('threat_intel')])} indicators")
print(f"  - IRIS case created: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []
containment_time = datetime.now().isoformat()

# 1. Isolate affected systems via CrowdStrike
print(f"\n[CONTAINMENT] Isolating affected systems...")
isolated_hosts = []
for system in affected_systems:
    try:
        result = crowdstrike.isolate_host(system['device_id'])
        if result:
            isolated_hosts.append(system['hostname'])
            containment_actions.append({
                'action': 'host_isolation',
                'target': system['hostname'],
                'device_id': system['device_id'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Isolated host: {system['hostname']}")
        else:
            print(f"   Failed to isolate host: {system['hostname']}")
    except Exception as e:
        print(f"   Error isolating {system['hostname']}: {e}")

# 2. Revoke elevated privileges and sessions
print(f"\n[CONTAINMENT] Revoking elevated privileges and sessions...")
revoked_privileges = []
for user in unique_users:
    try:
        # Force logout of elevated sessions
        result = shuffle.force_logout_user(user, f"Privilege escalation incident {incident_id}")
        if result:
            revoked_privileges.append(user)
            containment_actions.append({
                'action': 'privilege_revocation',
                'target': user,
                'reason': 'Privilege escalation detected - revoking elevated access',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Revoked privileges for: {user}")
        else:
            print(f"   Failed to revoke privileges for: {user}")
    except Exception as e:
        print(f"   Error revoking privileges for {user}: {e}")

# 3. Block suspicious processes and commands
print(f"\n[CONTAINMENT] Blocking suspicious processes and commands...")
blocked_processes = []
for indicator in privilege_escalation_indicators:
    try:
        if indicator.get('command'):
            # Extract process names to block
            process_names = re.findall(r'([a-zA-Z0-9\-_]+\.exe)', indicator['command'], re.IGNORECASE)
            for proc_name in process_names:
                if proc_name.lower() not in ['cmd.exe', 'powershell.exe', 'explorer.exe']:  # Don't block common legit processes
                    result = shuffle.block_process(proc_name, f"Privilege escalation incident {incident_id}")
                    if result:
                        blocked_processes.append(proc_name)
                        containment_actions.append({
                            'action': 'process_block',
                            'target': proc_name,
                            'reason': f'Suspicious privilege escalation activity on {indicator.get("host", "unknown")}',
                            'status': 'success',
                            'timestamp': containment_time
                        })
                        print(f"   Blocked process: {proc_name}")
    except Exception as e:
        print(f"   Error blocking processes: {e}")

# 4. Enable enhanced privilege monitoring
print(f"\n[CONTAINMENT] Enabling enhanced privilege monitoring...")
try:
    # Create Splunk alerts for privilege escalation
    alert_configs = [
        {
            'name': f'Enhanced Privilege Escalation Monitoring - {incident_id}',
            'search': 'index=* sourcetype=WinEventLog:Security EventCode=4672 OR EventCode=4688 | eval risk_score = case(match(CommandLine, "bypassuac|uac.*bypass"), 10, match(CommandLine, "token.*steal|token.*impersonate"), 10, match(CommandLine, "mimikatz|incognito"), 10, 1==1, 4) | where risk_score >= 8',
            'alert_type': 'real-time',
            'severity': 'critical'
        }
    ]

    for config in alert_configs:
        result = splunk.create_alert(config)
        if result:
            containment_actions.append({
                'action': 'enhanced_monitoring',
                'target': 'splunk_alert',
                'alert_name': config['name'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Created enhanced monitoring alert: {config['name']}")

    # Enable CrowdStrike enhanced privilege monitoring
    for system in affected_systems:
        result = crowdstrike.enable_privilege_monitoring(system['device_id'])
        if result:
            containment_actions.append({
                'action': 'enhanced_edr_privilege',
                'target': system['hostname'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Enabled enhanced privilege monitoring for: {system['hostname']}")

except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

# Update IRIS case with containment actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'containment_actions': containment_actions,
            'containment_time': containment_time,
            'isolated_hosts': isolated_hosts,
            'revoked_privileges': revoked_privileges,
            'blocked_processes': blocked_processes
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with containment details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} systems isolated")
print(f"  - {len(revoked_privileges)} user privileges revoked")
print(f"  - {len(blocked_processes)} suspicious processes blocked")
print(f"  - Enhanced privilege monitoring enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []
eradication_time = datetime.now().isoformat()

# 1. Terminate malicious processes via CrowdStrike
print(f"\n[ERADICATION] Terminating malicious processes...")
terminated_processes = []
for system in affected_systems:
    try:
        # Get running processes from CrowdStrike
        processes = crowdstrike.get_processes(system['device_id'])
        if processes:
            malicious_processes = []
            for proc in processes:
                proc_name = proc.get('name', '').lower()
                proc_cmd = proc.get('command_line', '').lower()

                # Check against known privilege escalation tools
                if any(tool in proc_cmd for tool in [
                    'mimikatz', 'incognito', 'tokenvator', 'psexec', 'cobalt strike',
                    'metasploit', 'empire', 'powersploit', 'uac bypass'
                ]) or any(pattern in proc_cmd for pattern in [
                    'bypassuac', 'uac bypass', 'token steal', 'token impersonate',
                    'duplicate token', 'seimpersonateprivilege', 'reflective loader'
                ]):
                    malicious_processes.append(proc)

            # Terminate malicious processes
            for proc in malicious_processes[:5]:  # Limit to top 5 per host
                result = crowdstrike.terminate_process(system['device_id'], proc['pid'])
                if result:
                    terminated_processes.append({
                        'host': system['hostname'],
                        'process': proc['name'],
                        'pid': proc['pid'],
                        'command': proc.get('command_line', '')
                    })
                    eradication_actions.append({
                        'action': 'process_termination',
                        'target': system['hostname'],
                        'process': proc['name'],
                        'pid': proc['pid'],
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Terminated process: {proc['name']} (PID: {proc['pid']}) on {system['hostname']}")
    except Exception as e:
        print(f"   Error terminating processes on {system['hostname']}: {e}")

# 2. Delete privilege escalation tools
print(f"\n[ERADICATION] Deleting privilege escalation tools...")
deleted_tools = []
for indicator in privilege_escalation_indicators:
    try:
        if indicator.get('command'):
            # Extract file paths from commands
            file_paths = re.findall(r'([C-Z]:[^\s"]*\.(exe|dll|bat|ps1|vbs|js))', indicator['command'], re.IGNORECASE)
            for file_path, ext in file_paths:
                result = crowdstrike.delete_file(indicator['host'], file_path)
                if result:
                    deleted_tools.append({
                        'host': indicator['host'],
                        'file': file_path
                    })
                    eradication_actions.append({
                        'action': 'tool_deletion',
                        'target': indicator['host'],
                        'file': file_path,
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Deleted tool: {file_path} on {indicator['host']}")

        # Clean up temp directories
        temp_dirs = [
            f"C:\\Users\\{indicator.get('user', 'unknown')}\\AppData\\Local\\Temp\\*",
            f"C:\\Users\\{indicator.get('user', 'unknown')}\\Downloads\\*",
            "C:\\Windows\\Temp\\*"
        ]
        for temp_dir in temp_dirs:
            result = crowdstrike.delete_file(indicator['host'], temp_dir)
            if result:
                print(f"   Cleaned temp directory: {temp_dir} on {indicator['host']}")
    except Exception as e:
        print(f"   Error deleting tools: {e}")

# 3. Reset user credentials and clear cached tokens
print(f"\n[ERADICATION] Resetting user credentials...")
reset_credentials = []
for user in unique_users:
    try:
        # Reset password
        result = shuffle.reset_user_password(user, f"Privilege escalation compromise - incident {incident_id}")
        if result:
            reset_credentials.append(user)
            eradication_actions.append({
                'action': 'credential_reset',
                'target': user,
                'reason': 'Privilege escalation detected - resetting compromised credentials',
                'status': 'success',
                'timestamp': eradication_time
            })
            print(f"   Reset credentials for: {user}")
        else:
            print(f"   Failed to reset credentials for: {user}")

        # Clear cached Kerberos tickets
        result = shuffle.clear_kerberos_tickets(user)
        if result:
            eradication_actions.append({
                'action': 'kerberos_cache_clear',
                'target': user,
                'status': 'success',
                'timestamp': eradication_time
            })
            print(f"   Cleared Kerberos cache for: {user}")
    except Exception as e:
        print(f"   Error resetting credentials for {user}: {e}")

# 4. Clean up privilege escalation persistence
print(f"\n[ERADICATION] Cleaning up privilege escalation persistence...")
cleaned_persistence = []
for system in affected_systems:
    try:
        # Check for scheduled tasks created for privilege escalation
        scheduled_tasks = crowdstrike.get_scheduled_tasks(system['device_id'])
        if scheduled_tasks:
            for task in scheduled_tasks:
                task_name = task.get('name', '').lower()
                task_cmd = task.get('command', '').lower()

                if any(suspicious in task_cmd for suspicious in [
                    'mimikatz', 'psexec', 'bypassuac', 'token', 'impersonate', 'elevated'
                ]):
                    result = crowdstrike.delete_scheduled_task(system['device_id'], task['name'])
                    if result:
                        cleaned_persistence.append({
                            'host': system['hostname'],
                            'task': task['name']
                        })
                        eradication_actions.append({
                            'action': 'scheduled_task_cleanup',
                            'target': system['hostname'],
                            'task': task['name'],
                            'status': 'success',
                            'timestamp': eradication_time
                        })
                        print(f"   Removed suspicious scheduled task: {task['name']} on {system['hostname']}")
    except Exception as e:
        print(f"   Error cleaning persistence on {system['hostname']}: {e}")

# 5. Verify eradication
print(f"\n[ERADICATION] Verifying eradication...")
verification_results = []
try:
    # Re-scan systems for remaining privilege escalation tools
    for system in affected_systems:
        scan_result = crowdstrike.scan_for_privilege_escalation(system['device_id'])
        if scan_result:
            verification_results.append({
                'host': system['hostname'],
                'scan_result': scan_result,
                'threats_found': len(scan_result.get('privilege_threats', []))
            })
            if scan_result.get('privilege_threats'):
                print(f"   Verification found {len(scan_result['privilege_threats'])} remaining privilege threats on {system['hostname']}")
            else:
                print(f"   Verification clean: {system['hostname']}")
        else:
            print(f"   Verification scan failed for: {system['hostname']}")
except Exception as e:
    print(f"   Error during verification: {e}")

# Update IRIS case with eradication actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'eradication_actions': eradication_actions,
            'eradication_time': eradication_time,
            'terminated_processes': terminated_processes,
            'deleted_tools': deleted_tools,
            'reset_credentials': reset_credentials,
            'cleaned_persistence': cleaned_persistence,
            'verification_results': verification_results
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with eradication details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(terminated_processes)} malicious processes terminated")
print(f"  - {len(deleted_tools)} privilege escalation tools deleted")
print(f"  - {len(reset_credentials)} user credentials reset")
print(f"  - {len(cleaned_persistence)} persistence mechanisms cleaned")
print(f"  - Verification completed for {len(verification_results)} systems")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []
recovery_time = datetime.now().isoformat()

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
reenabled_hosts = []
for system in affected_systems:
    if system['hostname'] in isolated_hosts:
        try:
            result = crowdstrike.unisolate_host(system['device_id'])
            if result:
                reenabled_hosts.append(system['hostname'])
                recovery_actions.append({
                    'action': 'host_unisolation',
                    'target': system['hostname'],
                    'device_id': system['device_id'],
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Re-enabled host: {system['hostname']}")
            else:
                print(f"   Failed to re-enable host: {system['hostname']}")
        except Exception as e:
            print(f"   Error re-enabling {system['hostname']}: {e}")

# 2. Restore normal user access and privileges
print(f"\n[RECOVERY] Restoring normal user access...")
restored_access = []
for user in unique_users:
    try:
        # Re-enable user account if it was disabled
        result = shuffle.enable_user_account(user, f"Recovery complete - privilege escalation incident {incident_id}")
        if result:
            restored_access.append(user)
            recovery_actions.append({
                'action': 'account_enable',
                'target': user,
                'reason': 'Recovery complete - privilege escalation incident resolved',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored access for: {user}")
        else:
            print(f"   Failed to restore access for: {user}")

        # Restore normal privilege levels
        result = shuffle.restore_user_privileges(user)
        if result:
            recovery_actions.append({
                'action': 'privilege_restore',
                'target': user,
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored normal privileges for: {user}")
    except Exception as e:
        print(f"   Error restoring access for {user}: {e}")

# 3. Validate system functionality and privilege controls
print(f"\n[RECOVERY] Validating system functionality...")
validation_results = []
for system in affected_systems:
    try:
        # Check system health via CrowdStrike
        health_check = crowdstrike.check_system_health(system['device_id'])
        if health_check:
            validation_results.append({
                'host': system['hostname'],
                'health_status': health_check.get('status', 'unknown'),
                'privilege_controls': health_check.get('privilege_status', 'unknown'),
                'connectivity': health_check.get('network_status', 'unknown')
            })

            if health_check.get('status') == 'healthy' and health_check.get('privilege_status') == 'normal':
                recovery_actions.append({
                    'action': 'health_validation',
                    'target': system['hostname'],
                    'status': 'success',
                    'privilege_status': 'normal',
                    'timestamp': recovery_time
                })
                print(f"   System and privilege controls validated: {system['hostname']}")
            else:
                print(f"   System health or privilege issues detected: {system['hostname']} - {health_check.get('issues', [])}")
        else:
            print(f"   Health check failed for: {system['hostname']}")
    except Exception as e:
        print(f"   Error validating {system['hostname']}: {e}")

# 4. Restore normal monitoring levels
print(f"\n[RECOVERY] Restoring normal monitoring levels...")
try:
    # Disable enhanced privilege escalation monitoring
    result = splunk.disable_alert(f'Enhanced Privilege Escalation Monitoring - {incident_id}')
    if result:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'splunk_alerts',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring levels")

    # Restore normal CrowdStrike monitoring
    for system in affected_systems:
        result = crowdstrike.restore_normal_monitoring(system['device_id'])
        if result:
            recovery_actions.append({
                'action': 'edr_monitoring_restore',
                'target': system['hostname'],
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored normal EDR monitoring for: {system['hostname']}")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 5. Monitor for residual privilege escalation attempts
print(f"\n[RECOVERY] Establishing post-recovery monitoring...")
try:
    # Create follow-up monitoring for privilege escalation
    followup_alerts = [
        {
            'name': f'Post-Recovery Privilege Escalation Monitoring - {incident_id}',
            'search': f'index=* host IN ({",".join([f'"{s["hostname"]}"' for s in affected_systems])}) sourcetype=WinEventLog:Security EventCode=4672 | eval risk_score = case(match(CommandLine, "bypassuac|token.*steal|mimikatz"), 10, match(CommandLine, "runas.*admin|psexec"), 8, 1==1, 4) | where risk_score >= 8',
            'alert_type': 'scheduled',
            'severity': 'high',
            'schedule': '0 */6 * * *'  # Every 6 hours
        }
    ]

    for alert_config in followup_alerts:
        result = splunk.create_alert(alert_config)
        if result:
            recovery_actions.append({
                'action': 'followup_monitoring',
                'target': 'splunk_alert',
                'alert_name': alert_config['name'],
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Created follow-up monitoring: {alert_config['name']}")

except Exception as e:
    print(f"   Error establishing follow-up monitoring: {e}")

# Update IRIS case with recovery actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'recovery_actions': recovery_actions,
            'recovery_time': recovery_time,
            'reenabled_hosts': reenabled_hosts,
            'restored_access': restored_access,
            'validation_results': validation_results
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with recovery details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(reenabled_hosts)} systems re-enabled")
print(f"  - {len(restored_access)} user accounts restored")
print(f"  - System validation completed for {len(validation_results)} systems")
print(f"  - Normal monitoring levels restored")
print(f"  - Post-recovery monitoring established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': f'Privilege Escalation Incident Report - {len(privilege_escalation_indicators)} indicators',
        'detection_time': detection_time,
        'closure_time': closure_time,
        'severity': 'CRITICAL',
        'tactic': 'Privilege Escalation',
        'technique': 'Abuse Elevation Control Mechanism (T1548)',
        'summary': {
            'affected_users': len(unique_users),
            'affected_systems': len(affected_systems),
            'indicators_detected': len(privilege_escalation_indicators),
            'hosts_isolated': len(isolated_hosts),
            'privileges_revoked': len(revoked_privileges),
            'processes_terminated': len(terminated_processes),
            'tools_deleted': len(deleted_tools),
            'credentials_reset': len(reset_credentials)
        },
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': closure_time
        },
        'actions_taken': {
            'detection': privilege_escalation_indicators[:10],  # Top 10 indicators
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'threat_intelligence': [i.get('threat_intel') for i in privilege_escalation_indicators if i.get('threat_intel')],
        'recommendations': [
            'Implement UAC restrictions and application whitelisting',
            'Deploy privilege monitoring and anomaly detection',
            'Regular credential rotation and access reviews',
            'Enhanced logging for privilege operations',
            'User training on privilege escalation risks'
        ]
    }

    # Save report to file
    report_filename = f"privilege_escalation_report_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'report_generation',
        'target': report_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Generated incident report: {report_filename}")

except Exception as e:
    print(f"   Error generating report: {e}")

# 2. Document lessons learned
print(f"\n[POST-INCIDENT] Documenting lessons learned...")
try:
    lessons_learned = {
        'incident_id': incident_id,
        'what_went_well': [
            'Rapid detection of privilege escalation attempts',
            'Effective isolation of compromised systems',
            'Comprehensive eradication of privilege escalation tools',
            'Successful credential reset and privilege restoration'
        ],
        'what_could_improve': [
            'Earlier detection of UAC bypass techniques',
            'Enhanced monitoring of token manipulation',
            'Automated privilege escalation prevention',
            'Better user training on elevation risks'
        ],
        'key_findings': [
            f'Privilege escalation affected {len(unique_users)} users across {len(affected_systems)} systems',
            f'Most common technique: {max([i.get("type", "unknown") for i in privilege_escalation_indicators], key=[i.get("type", "unknown") for i in privilege_escalation_indicators].count)}',
            'Threat intelligence enriched detection for key privilege escalation tools',
            'Automated response prevented lateral movement and data exfiltration'
        ],
        'preventive_measures': [
            'Implement just-in-time access and privilege management',
            'Deploy advanced endpoint protection with privilege monitoring',
            'Regular security assessments and penetration testing',
            'Enhanced logging and monitoring of privilege operations',
            'Security awareness training focused on privilege escalation'
        ]
    }

    # Save lessons learned
    lessons_filename = f"privilege_escalation_lessons_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(lessons_filename, 'w') as f:
        json.dump(lessons_learned, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'lessons_learned',
        'target': lessons_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Documented lessons learned: {lessons_filename}")

except Exception as e:
    print(f"   Error documenting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_measures = []

    # Update Splunk correlation rules for privilege escalation
    updated_rules = splunk.update_correlation_rules([
        {
            'name': 'Enhanced Privilege Escalation Detection',
            'search': 'index=* sourcetype=WinEventLog:Security EventCode=4672 OR EventCode=4688 | eval risk_score = case(match(CommandLine, "bypassuac|uac.*bypass|token.*steal|token.*impersonate"), 10, match(CommandLine, "mimikatz|incognito|psexec|runas.*admin"), 9, match(CommandLine, "schtasks.*highest|at.*interactive"), 8, 1==1, 4) | where risk_score >= 8',
            'alert_threshold': 3,
            'time_window': '10m'
        }
    ])
    if updated_rules:
        preventive_measures.append('Updated Splunk privilege escalation rules')
        print(f"   Updated Splunk correlation rules for privilege escalation detection")

    # Enhance CrowdStrike prevention policies
    cs_policies = crowdstrike.update_prevention_policies({
        'uac_protection': 'strict',
        'privilege_escalation_prevention': 'enabled',
        'token_manipulation_detection': 'enabled',
        'process_injection_prevention': 'enabled'
    })
    if cs_policies:
        preventive_measures.append('Enhanced CrowdStrike privilege policies')
        print(f"   Enhanced CrowdStrike prevention policies")

    # Schedule privilege escalation security training
    training_scheduled = shuffle.schedule_security_training(
        users=list(unique_users),
        topic='Privilege Escalation Awareness and Prevention',
        incident_reference=incident_id
    )
    if training_scheduled:
        preventive_measures.append('Scheduled privilege escalation training')
        print(f"   Scheduled security awareness training on privilege escalation")

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Share threat intelligence
print(f"\n[POST-INCIDENT] Sharing threat intelligence...")
try:
    shared_intel = []
    for indicator in privilege_escalation_indicators:
        if indicator.get('threat_intel'):
            # Share with MISP
            result = misp.share_indicator(indicator, incident_id)
            if result:
                shared_intel.append(f"MISP: {indicator.get('type', 'unknown')}")
                print(f"   Shared indicator with MISP: {indicator.get('type', 'unknown')}")

    if shared_intel:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': shared_intel,
            'status': 'success',
            'timestamp': closure_time
        })

except Exception as e:
    print(f"   Error sharing threat intelligence: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    if incident_id:
        closure_data = {
            'status': 'closed',
            'closure_time': closure_time,
            'closure_reason': 'Privilege escalation incident successfully contained, eradicated, and recovered',
            'post_incident_actions': post_incident_actions,
            'final_assessment': {
                'threat_contained': len(isolated_hosts) > 0,
                'threat_eradicated': len(terminated_processes) > 0 or len(deleted_tools) > 0,
                'systems_recovered': len(reenabled_hosts) > 0,
                'preventive_measures': len(preventive_measures) > 0
            }
        }

        result = iris.close_case(incident_id, closure_data)
        if result:
            post_incident_actions.append({
                'action': 'case_closure',
                'target': incident_id,
                'status': 'success',
                'timestamp': closure_time
            })
            print(f"   Closed IRIS case: {incident_id}")
        else:
            print(f"   Failed to close IRIS case: {incident_id}")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated")
print(f"  - Lessons learned documented")
print(f"  - {len(preventive_measures)} preventive measures implemented")
print(f"  - Threat intelligence shared with {len(shared_intel)} platforms")
print(f"  - Incident case closed: {incident_id}")

print(f"\n Privilege Escalation Incident Response Complete")
print(f"   Total duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")
print(f"   Response effectiveness: {'HIGH' if len(isolated_hosts) > 0 and len(terminated_processes) > 0 else 'MEDIUM'}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
